In [ ]:
def optimal_fixed_size_tiles(
    image,
    mask,
    tile_size,
    overlap_pixels=0,
    mask_pixel_threshold=0.0,
    background_value=0.0,
):
    """
    Generate fixed-size square tiles (tile_size x tile_size) that best cover
    a given mask, using a greedy selection of tile positions + fallback for
    isolated leftovers.

    Then deduplicate tiles that cover exactly the same set of mask pixels,
    keeping only the best-centered tile in each group.

    Args:
        image (np.ndarray): Original RGB image, float32 [0,1], shape (H, W, C).
        mask (np.ndarray): Binary mask (0/1, 0/255, bool), shape (H, W).
        tile_size (int): Size of the square crops.
        overlap_pixels (int): Overlap between neighboring candidate tiles.
                              Step = tile_size - overlap_pixels.
        mask_pixel_threshold (float): Minimum fraction of mask pixels inside a tile
                                      (relative to tile_size * tile_size) to even
                                      consider that tile in the greedy stage.
        background_value (float): Value to pad outside-image regions with.

    Returns:
        list of tuples:
            (image_tile, mask_tile, (row_start, col_start))
    """

    H, W = image.shape[:2]

    step = tile_size - overlap_pixels
    if step <= 0:
        raise ValueError("overlap_pixels must be < tile_size (step must be > 0).")

    mask_bin = (mask > 0).astype(np.uint8)

    if mask_bin.sum() == 0:
        return []

    if tile_size >= H:
        r_starts = [0]
    else:
        r_starts = list(range(0, H - tile_size + 1, step))
        if r_starts[-1] != H - tile_size:
            r_starts.append(H - tile_size)

    if tile_size >= W:
        c_starts = [0]
    else:
        c_starts = list(range(0, W - tile_size + 1, step))
        if c_starts[-1] != W - tile_size:
            c_starts.append(W - tile_size)

    candidate_tiles = []
    tile_area = float(tile_size * tile_size)

    for r in r_starts:
        for c in c_starts:
            copy_r_start = max(0, r)
            copy_r_end   = min(H, r + tile_size)
            copy_c_start = max(0, c)
            copy_c_end   = min(W, c + tile_size)

            if copy_r_start >= copy_r_end or copy_c_start >= copy_c_end:
                continue

            tile_mask = mask_bin[copy_r_start:copy_r_end, copy_c_start:copy_c_end]
            mask_pixels = tile_mask.sum()
            frac_mask = mask_pixels / tile_area

            if mask_pixels > 0 and frac_mask >= mask_pixel_threshold:
                candidate_tiles.append((mask_pixels, r, c))

    tiles = []
    selected_positions = set()
    uncovered = mask_bin.copy()

    if candidate_tiles:
        candidate_tiles.sort(key=lambda x: x[0], reverse=True)

        for mask_pixels, r, c in candidate_tiles:
            copy_r_start = max(0, r)
            copy_r_end   = min(H, r + tile_size)
            copy_c_start = max(0, c)
            copy_c_end   = min(W, c + tile_size)

            tile_uncovered = uncovered[copy_r_start:copy_r_end, copy_c_start:copy_c_end]
            new_coverage = tile_uncovered.sum()

            if new_coverage == 0:
                continue

            img_tile = np.full(
                (tile_size, tile_size, image.shape[2]),
                background_value,
                dtype=image.dtype
            )
            mask_tile = np.zeros((tile_size, tile_size), dtype=mask.dtype)

            paste_r_start = copy_r_start - r
            paste_c_start = copy_c_start - c
            paste_r_end   = paste_r_start + (copy_r_end - copy_r_start)
            paste_c_end   = paste_c_start + (copy_c_end - copy_c_start)

            img_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end, :] = \
                image[copy_r_start:copy_r_end, copy_c_start:copy_c_end, :]
            mask_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end] = \
                mask[copy_r_start:copy_r_end, copy_c_start:copy_c_end]

            tiles.append((img_tile, mask_tile, (r, c)))
            selected_positions.add((r, c))

            uncovered[copy_r_start:copy_r_end, copy_c_start:copy_c_end] = 0

            if uncovered.sum() == 0:
                break

    if uncovered.sum() > 0:
        num_labels, labels = cv2.connectedComponents(uncovered.astype(np.uint8))

        for label_id in range(1, num_labels):
            ys, xs = np.where(labels == label_id)
            if ys.size == 0:
                continue

            center_r = int(ys.mean())
            center_c = int(xs.mean())

            if tile_size >= H:
                r_forced = 0
            else:
                r_forced = center_r - tile_size // 2
                r_forced = max(0, min(r_forced, H - tile_size))

            if tile_size >= W:
                c_forced = 0
            else:
                c_forced = center_c - tile_size // 2
                c_forced = max(0, min(c_forced, W - tile_size))

            if (r_forced, c_forced) in selected_positions:
                continue

            copy_r_start = max(0, r_forced)
            copy_r_end   = min(H, r_forced + tile_size)
            copy_c_start = max(0, c_forced)
            copy_c_end   = min(W, c_forced + tile_size)

            img_tile = np.full(
                (tile_size, tile_size, image.shape[2]),
                background_value,
                dtype=image.dtype
            )
            mask_tile = np.zeros((tile_size, tile_size), dtype=mask.dtype)

            paste_r_start = copy_r_start - r_forced
            paste_c_start = copy_c_start - c_forced
            paste_r_end   = paste_r_start + (copy_r_end - copy_r_start)
            paste_c_end   = paste_c_start + (copy_c_end - copy_c_start)

            img_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end, :] = \
                image[copy_r_start:copy_r_end, copy_c_start:copy_c_end, :]
            mask_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end] = \
                mask[copy_r_start:copy_r_end, copy_c_start:copy_c_end]

            tiles.append((img_tile, mask_tile, (r_forced, c_forced)))
            selected_positions.add((r_forced, c_forced))

    if not tiles:
        return []

    dedup_map = {}
    for idx, (_, _, (r, c)) in enumerate(tiles):
        copy_r_start = max(0, r)
        copy_r_end   = min(H, r + tile_size)
        copy_c_start = max(0, c)
        copy_c_end   = min(W, c + tile_size)

        tile_mask = (mask[copy_r_start:copy_r_end, copy_c_start:copy_c_end] > 0)
        ys, xs = np.where(tile_mask)

        if ys.size == 0:
            global_idxs = ()
            centroid_r = centroid_c = None
        else:
            global_ys = ys + copy_r_start
            global_xs = xs + copy_c_start
            global_idxs = tuple(sorted(global_ys * W + global_xs))

            centroid_r = float(global_ys.mean())
            centroid_c = float(global_xs.mean())

        tile_center_r = r + tile_size / 2.0
        tile_center_c = c + tile_size / 2.0

        if centroid_r is None:
            score = 0.0
        else:
            dr = tile_center_r - centroid_r
            dc = tile_center_c - centroid_c
            score = dr * dr + dc * dc

        if global_idxs not in dedup_map or score < dedup_map[global_idxs][0]:
            dedup_map[global_idxs] = (score, idx)

    final_tiles = [tiles[v[1]] for v in dedup_map.values()]

    return final_tiles